# Cwiczenie 3 - Pandas (seria 3 z PDSH) — wersja rozszerzona v2

Zestaw jest celowo ustawiony tak, aby stopniowo budowac kompetencje potrzebne do projektu
podsumowujacego 1-4: najpierw czyszczenie danych i selekcja, potem merge/groupby/pivot,
a na koncu proste elementy optymalizacji w Pandas.

Nowe zadania dopinaja luki z calego wykladu 3:
- alignment po indeksie,
- concat z `keys`,
- audyt merge przez `validate` i `indicator`,
- `transform` oraz `filter`,
- `pivot_table` z `fill_value` i `margins`,
- regexy w `.str`,
- lag/shift w szeregach czasowych,
- mini-benchmarki.

W zadaniach z sekcja **Wymagane metody** postaraj sie nie omijac wskazanych narzedzi.


## Proponowany tryb pracy

**CORE (obowiązkowe minimum):**
- 03.01–03.09
- 03.11.1
- 03.07.2 (merge audit) warto zrobić wspólnie z prowadzącym

**ROZSZERZENIE / homework / dla chętnych:**
- 03.10.2 (regex)
- 03.11.2 (lag / shift)
- 03.12.2–03.12.3 (mini-benchmarki)
- sekcja o `category`

Dzięki temu notebook nie przytłacza na pierwszym przejściu, ale nadal zachowuje pełną ścieżkę rozwoju pod projekt.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd


def _ensure_core_import() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / "pack_pandas_multiindex",
        Path.cwd() / "cwiczenie_3_numpy_full_pack" / "pack_pandas_multiindex",
        Path("/mnt/data/3_extracted/cwiczenie_3_numpy_full_pack/pack_pandas_multiindex"),
        Path("/mnt/data/pandas_exercises_v2"),
    ]
    for candidate in candidates:
        if (candidate / "projekt_1_4_bikeshare_core.py").exists():
            candidate_str = str(candidate.resolve())
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return candidate
    raise FileNotFoundError("Nie znaleziono common/projekt_1_4_bikeshare_core.py")


CORE_PATH = _ensure_core_import()

from projekt_1_4_bikeshare_core import make_bikeshare_data, revenue_loop, time_one

pd.set_option("display.max_columns", 30)
np.random.seed(0)

trips, stations, weather = make_bikeshare_data(n_trips=5000, n_stations=18, n_days=14, seed=0)

# pomocnicze kopie robocze
trips_small = trips.copy()
stations_small = stations.copy()
weather_small = weather.copy()

print(f"Core imported from: {CORE_PATH}")
print(trips_small.head(2))
print(stations_small.head(2))
print(weather_small.head(2))

## Mapa tego notebooka

Ten zestaw najlepiej działa w trzech krokach:

1. **Oczyść i opisz dane**  
   (`select`, `assign`, missing values, stringi)

2. **Połącz dane i policz agregaty**  
   (`merge`, `groupby`, `pivot_table`, `transform`, `validate`)

3. **Zamień tabelę w prostą analizę**  
   (`rolling`, `lag`, krótkie wnioski analityczne)

Na zajęciach warto pilnować jednej zasady:
**najpierw liczba wierszy i klucze, potem dopiero ładne wyniki.**


## 03.01 - Introducing Pandas Objects


### Cwiczenie 3.01.1 - Seria KPI z DataFrame

Napisz funkcje, ktora zbuduje obiekt `pd.Series` z podstawowymi KPI:
- `n_trips`
- `n_unique_bikes`
- `mean_duration_min`
- `share_member`


In [ ]:
def trip_kpi_series(trips: pd.DataFrame) -> pd.Series:
    duration_min = (
        pd.to_datetime(trips["end_ts"]) - pd.to_datetime(trips["start_ts"])
    ).dt.total_seconds() / 60.0
    return pd.Series(
        {
            "n_trips": int(len(trips)),
            "n_unique_bikes": int(trips["bike_id"].nunique()),
            "mean_duration_min": float(duration_min.mean()),
            "share_member": float(trips["user_type"].eq("member").mean()),
        }
    )


kpi = trip_kpi_series(trips_small)
assert isinstance(kpi, pd.Series)
assert list(kpi.index) == ["n_trips", "n_unique_bikes", "mean_duration_min", "share_member"]
assert int(kpi["n_trips"]) == len(trips_small)
assert 0.0 <= float(kpi["share_member"]) <= 1.0
kpi


## 03.02 - Data Indexing and Selection


### Cwiczenie 3.02.1 - Selekcja przejazdow w godzinach szczytu

Zwracaj tylko przejazdy rozpoczete:
- rano: od 7:00 do 9:59
- wieczorem: od 16:00 do 18:59

Dodaj kolumne `start_hour`.


In [ ]:
def select_rush_hours(trips: pd.DataFrame) -> pd.DataFrame:
    out = trips.copy()
    out["start_hour"] = pd.to_datetime(out["start_ts"]).dt.hour
    mask = out["start_hour"].between(7, 9) | out["start_hour"].between(16, 18)
    return out.loc[mask].copy()


rush = select_rush_hours(trips_small)
assert isinstance(rush, pd.DataFrame)
assert "start_hour" in rush.columns
assert rush["start_hour"].between(7, 9).all() or rush["start_hour"].between(16, 18).any()
assert set(rush["start_hour"].unique()).issubset(set([7, 8, 9, 16, 17, 18]))
rush.head()


## 03.03 - Operations in Pandas


### Cwiczenie 3.03.1 - Dodaj cechy obliczone wektorowo

Utworz kopie `trips` i dodaj kolumny:
- `duration_min` = roznica `end_ts - start_ts` w minutach
- `is_member` = czy `user_type == "member"`
- `revenue_est` wedlug prostego cennika:
  - member: 1.0 + 0.12 * max(duration_min - 5, 0)
  - casual: 2.5 + 0.23 * max(duration_min - 5, 0)

Bez petli po wierszach.


In [ ]:
def add_trip_features(trips: pd.DataFrame) -> pd.DataFrame:
    out = trips.copy()
    out["duration_min"] = (
        pd.to_datetime(out["end_ts"]) - pd.to_datetime(out["start_ts"])
    ).dt.total_seconds() / 60.0
    out["is_member"] = out["user_type"].eq("member")

    base = np.where(out["is_member"], 1.0, 2.5)
    per_min = np.where(out["is_member"], 0.12, 0.23)
    out["revenue_est"] = base + per_min * np.maximum(out["duration_min"].to_numpy() - 5.0, 0.0)

    out["date"] = pd.to_datetime(out["start_ts"]).dt.floor("D")
    return out


feat = add_trip_features(trips_small)
assert {"duration_min", "is_member", "revenue_est"}.issubset(feat.columns)
assert feat["duration_min"].ge(0).all()
assert feat["is_member"].dtype == bool
assert feat["revenue_est"].ge(0).all()
feat[["duration_min", "is_member", "revenue_est"]].head()


### Cwiczenie 3.03.2 - Alignment po indeksie: dzienne cechy popytu i pogody

Zbuduj tabele indeksowana po dacie z kolumnami:
- `n_trips`
- `mean_temp_c`
- `total_precip_mm`
- `trips_per_mm`

**Wymagane metody**
- ustawienie indeksu czasowego,
- `resample('D')`,
- polaczenie przez alignment indeksu (`join` albo `concat`), **bez `merge()`**,
- ochrona przed dzieleniem przez zero przy `trips_per_mm`.


In [ ]:
def daily_weather_alignment(trips: pd.DataFrame, weather: pd.DataFrame) -> pd.DataFrame:
    daily_trips = (
        trips.set_index(pd.to_datetime(trips["start_ts"]))
        .resample("D")
        .size()
        .rename("n_trips")
    )

    daily_weather = (
        weather.set_index("timestamp_hour")
        .sort_index()
        .resample("D")
        .agg(
            mean_temp_c=("temp_c", "mean"),
            total_precip_mm=("precip_mm", "sum"),
        )
    )

    out = pd.concat([daily_trips, daily_weather], axis=1)
    out["trips_per_mm"] = out["n_trips"] / out["total_precip_mm"].replace(0, np.nan)
    return out


aligned = daily_weather_alignment(trips_small, weather_small)
assert isinstance(aligned.index, pd.DatetimeIndex)
assert {"n_trips", "mean_temp_c", "total_precip_mm", "trips_per_mm"}.issubset(aligned.columns)
assert int(aligned["n_trips"].sum()) == len(trips_small)
assert aligned.index.is_monotonic_increasing
aligned.head()


## 03.04 - Missing Values


### Cwiczenie 3.04.1 - Uzupelnij braki w pogodzie

Uzupelnij:
- `temp_c` przez interpolacje po czasie,
- `precip_mm` wartoscia 0.

Zwracany DataFrame ma nie miec brakow w tych dwoch kolumnach.


In [ ]:
def fill_weather_missing(weather: pd.DataFrame) -> pd.DataFrame:
    out = weather.sort_values("timestamp_hour").copy().set_index("timestamp_hour")
    out["temp_c"] = out["temp_c"].interpolate(method="time").ffill().bfill()
    out["precip_mm"] = out["precip_mm"].fillna(0.0)
    return out.reset_index()


weather_filled = fill_weather_missing(weather_small)
assert isinstance(weather_filled, pd.DataFrame)
assert weather_filled["temp_c"].isna().sum() == 0
assert weather_filled["precip_mm"].isna().sum() == 0
weather_filled.head()


### Cwiczenie 3.04.2 - Audit brakow danych

Zbuduj tabele jakosci danych dla dowolnego DataFrame, zawierajaca kolumny:
- `variable`
- `n_missing`
- `pct_missing`
- `dtype`

Posortuj wynik malejaco po `n_missing`.

**Wymagane metody**
- `isna()`,
- agregacje po kolumnach,
- zbudowanie nowego `DataFrame` z podsumowaniem.


In [ ]:
def missing_audit(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(
        {
            "variable": df.columns,
            "n_missing": df.isna().sum().to_numpy(),
            "pct_missing": df.isna().mean().to_numpy(),
            "dtype": df.dtypes.astype(str).to_numpy(),
        }
    )
    return out.sort_values(["n_missing", "variable"], ascending=[False, True], ignore_index=True)


audit = missing_audit(weather_small)
assert list(audit.columns) == ["variable", "n_missing", "pct_missing", "dtype"]
assert audit["n_missing"].is_monotonic_decreasing
assert set(audit["variable"]) == set(weather_small.columns)
audit


## 03.05 - Hierarchical Indexing


### Cwiczenie 3.05.1 - MultiIndex: popyt per dzielnica i godzina

Zbuduj `pd.Series` z indeksem wielopoziomowym `(district, hour)`,
ktora zawiera liczbe przejazdow startujacych z danej dzielnicy o danej godzinie.


In [ ]:
def demand_multiindex(trips: pd.DataFrame, stations: pd.DataFrame) -> pd.Series:
    merged = trips.merge(
        stations[["station_id", "district"]],
        left_on="true_start_station_id",
        right_on="station_id",
        how="left",
    )
    merged["hour"] = pd.to_datetime(merged["start_ts"]).dt.hour
    out = merged.groupby(["district", "hour"]).size().sort_index()
    out.name = "n_trips"
    return out


mi = demand_multiindex(trips_small, stations_small)
assert isinstance(mi, pd.Series)
assert mi.index.nlevels == 2
assert mi.sum() == len(trips_small)
mi.head(10)


### Cwiczenie 3.05.2 - MultiIndex: przekroje (`.xs`) i slicing godzin

Mamy juz serie `mi` z indeksem `(district, hour)`. Teraz przećwicz:

1. Pobieranie przekroju dla jednej dzielnicy (profil godzinowy).
2. Wycinanie zakresu godzin dla wszystkich dzielnic przez `pd.IndexSlice`.

Uwaga: slicing po poziomie `hour` wymaga, aby indeks byl posortowany (`sort_index()`).


In [ ]:
def district_profile_hours(mi: pd.Series, district: str) -> pd.Series:
    out = mi.xs(district, level="district").sort_index()
    out.index.name = "hour"
    return out


def slice_hours(mi: pd.Series, hour_start: int, hour_end: int) -> pd.Series:
    mi_sorted = mi.sort_index()
    idx = pd.IndexSlice
    return mi_sorted.loc[idx[:, hour_start:hour_end]]


some_district = mi.index.get_level_values("district")[0]

prof = district_profile_hours(mi, some_district)
sub = slice_hours(mi, 7, 9)

assert isinstance(prof, pd.Series)
assert int(prof.sum()) == int(mi.xs(some_district, level="district").sum())
assert set(sub.index.get_level_values("hour")).issubset(set([7, 8, 9]))

prof.head(), sub.head()


### Cwiczenie 3.05.3 - MultiIndex: pelna siatka + `unstack` (heatmap)

W praktyce nie wszystkie kombinacje `(district, hour)` muszą występować w danych.
Zbuduj więc **pelna siatke** (district x 24 godziny), wypelnij braki zerami,
a nastepnie przejdz do postaci macierzy przez `unstack('hour')`.


In [ ]:
def demand_heatmap(mi: pd.Series, districts: list[str]) -> pd.DataFrame:
    full_idx = pd.MultiIndex.from_product([districts, range(24)], names=["district", "hour"])
    mi_full = mi.reindex(full_idx, fill_value=0).astype(int).sort_index()
    df = mi_full.unstack("hour").sort_index()
    return df.sort_index(axis=1)


districts = sorted(stations_small["district"].unique().tolist())
heat = demand_heatmap(mi, districts)

assert isinstance(heat, pd.DataFrame)
assert heat.shape == (len(districts), 24)
assert int(heat.to_numpy().sum()) == len(trips_small)

heat.head()


## 03.06 - Concat and Append


### Cwiczenie 3.06.1 - Polacz tygodniowe paczki

Podziel `trips_small` na 3 czesci i napisz funkcje, ktora zlacza je z powrotem
tak, aby zachowac wszystkie wiersze i zresetowac indeks.


In [ ]:
parts = [
    trips_small.iloc[:1500].copy(),
    trips_small.iloc[1500:3200].copy(),
    trips_small.iloc[3200:].copy(),
]

def concat_trip_parts(parts: list[pd.DataFrame]) -> pd.DataFrame:
    return pd.concat(parts, ignore_index=True)


joined = concat_trip_parts(parts)
assert isinstance(joined, pd.DataFrame)
assert len(joined) == len(trips_small)
assert joined.index.equals(pd.RangeIndex(len(joined)))
joined.head()


### Cwiczenie 3.06.2 - `concat` z kluczami: zachowaj pochodzenie partii

Polacz te same paczki `parts`, ale tym razem zachowaj informacje o zrodlowej partii
w zewnetrznym poziomie indeksu.

Wynik ma miec `MultiIndex` o poziomach:
- `batch`
- `row_id`

**Wymagane metody**
- `pd.concat(..., keys=..., names=...)`


In [ ]:
def concat_trip_parts_with_keys(parts: list[pd.DataFrame]) -> pd.DataFrame:
    keys = [f"part_{i + 1}" for i in range(len(parts))]
    return pd.concat(parts, keys=keys, names=["batch", "row_id"])


joined_keys = concat_trip_parts_with_keys(parts)
assert isinstance(joined_keys.index, pd.MultiIndex)
assert joined_keys.index.names == ["batch", "row_id"]
assert joined_keys.index.get_level_values("batch").nunique() == len(parts)
assert len(joined_keys) == len(trips_small)
joined_keys.head()


## 03.07 - Merge and Join


### Checkpoint 3.07.A — zanim zrobisz `merge`, nazwij relację

To jest krótki, ale bardzo ważny nawyk analityczny.

Zanim napiszesz `merge(...)`, odpowiedz:
- jaka jest relacja między `trips` i `stations`?
- jaka jest relacja między `trips` i pogodą po zaokrągleniu czasu do godziny?
- dlaczego tutaj sensowny jest `left join`, a nie `inner join`?

Nie chodzi o teorię dla teorii — chodzi o to, żeby po scaleniu nie zgubić ani nie rozmnożyć wierszy.


In [ ]:
CARDINALITY_STATIONS = "many_to_one"
CARDINALITY_WEATHER = "many_to_one"
WHY_LEFT_JOIN = (
    "Tabela `trips` jest tutaj tabelą główną i każda obserwacja opisuje pojedynczy przejazd. "
    "Do jednej stacji może odnosić się wiele przejazdów, a po zaokrągleniu czasu do godziny "
    "wiele przejazdów może wskazywać ten sam rekord pogody, więc w obu przypadkach relacja ma "
    "charakter `many_to_one`. "
    "Używamy `left join`, bo chcemy zachować pełną liczbę przejazdów nawet wtedy, gdy metadane stacji "
    "albo pogody okażą się niekompletne; `inner join` mógłby cicho wyrzucić część danych."
)

assert "many_to_one" in CARDINALITY_STATIONS.replace("-", "_").lower()
assert "many_to_one" in CARDINALITY_WEATHER.replace("-", "_").lower()
assert isinstance(WHY_LEFT_JOIN, str) and len(WHY_LEFT_JOIN.strip()) >= 100
print("Checkpoint 3.07.A ✅")


### Cwiczenie 3.07.1 - Polacz przejazdy, stacje i pogode

Zbuduj DataFrame z:
- przejazdami,
- metadanymi stacji startowej (`station_name`, `district`, `capacity`),
- pogoda dla godziny startu (`temp_c`, `precip_mm`, `wind_kmh`).

Wskazowka: pogode dolacz po `start_ts.dt.floor('h')`.


In [ ]:
def merge_trip_context(
    trips: pd.DataFrame,
    stations: pd.DataFrame,
    weather: pd.DataFrame,
) -> pd.DataFrame:
    out = trips.copy()
    out["start_hour_ts"] = pd.to_datetime(out["start_ts"]).dt.floor("h")

    stations_meta = stations[["station_id", "station_name", "district", "capacity"]].copy()
    out = out.merge(
        stations_meta,
        left_on="true_start_station_id",
        right_on="station_id",
        how="left",
    )
    out = out.merge(
        weather,
        left_on="start_hour_ts",
        right_on="timestamp_hour",
        how="left",
    )
    return out.drop(columns=["station_id"])


merged = merge_trip_context(trips_small, stations_small, weather_filled)
required = {"station_name", "district", "capacity", "temp_c", "precip_mm", "wind_kmh"}
assert required.issubset(merged.columns)
assert len(merged) == len(trips_small)
merged.head()


### Cwiczenie 3.07.2 - Merge audit przez `validate` i `indicator`

Zdarza sie, ze w logach pojawia sie bledny `station_id`. Stworz kopie `trips`,
ustaw w pierwszym wierszu `true_start_station_id = -999`, a nastepnie wykonaj merge audytowy z tabela stacji.

Wynik ma:
- zachowac wszystkie wiersze (`how='left'`),
- sprawdzic relacje `many_to_one`,
- dodac kolumne pochodzenia przez `indicator=True`.

Zwracaj:
1. scalony `DataFrame`,
2. `Series` z liczebnoscia `_merge`.

**Wymagane metody**
- `merge`,
- `validate='many_to_one'`,
- `indicator=True`.


In [ ]:
def audited_station_merge(trips: pd.DataFrame, stations: pd.DataFrame):
    stations_meta = stations[["station_id", "station_name", "district", "capacity"]].copy()
    out = trips.merge(
        stations_meta,
        left_on="true_start_station_id",
        right_on="station_id",
        how="left",
        validate="many_to_one",
        indicator=True,
    )
    counts = out["_merge"].value_counts().sort_index()
    return out, counts


trips_probe = trips_small.copy()
trips_probe.loc[0, "true_start_station_id"] = -999

merge_audit_df, merge_audit_counts = audited_station_merge(trips_probe, stations_small)
assert "_merge" in merge_audit_df.columns
assert merge_audit_counts.index.isin(["left_only", "right_only", "both"]).all()
assert int(merge_audit_counts.get("left_only", 0)) >= 1
assert len(merge_audit_df) == len(trips_probe)
merge_audit_counts, merge_audit_df.head()


## 03.08 - Aggregation and Grouping


### Cwiczenie 3.08.1 - Dzienne KPI stacji

Dla kazdej pary `(date, true_start_station_id)` policz:
- `n_trips`
- `mean_duration_min`
- `total_revenue`

Wynik powinien byc DataFrame z kolumnami o takich nazwach.


In [ ]:
def station_daily_kpis(merged: pd.DataFrame) -> pd.DataFrame:
    out = merged.copy()
    out["date"] = pd.to_datetime(out["start_ts"]).dt.floor("D")
    return (
        out.groupby(["date", "true_start_station_id"], as_index=False)
        .agg(
            n_trips=("trip_id", "size"),
            mean_duration_min=("duration_min", "mean"),
            total_revenue=("revenue_est", "sum"),
        )
        .sort_values(["date", "true_start_station_id"])
    )


merged_feat = add_trip_features(merged)
kpis = station_daily_kpis(merged_feat)
assert isinstance(kpis, pd.DataFrame)
assert {"date", "true_start_station_id", "n_trips", "mean_duration_min", "total_revenue"}.issubset(kpis.columns)
assert kpis["n_trips"].ge(1).all()
kpis.head()


### Cwiczenie 3.08.2 - `transform` i `filter` po grupach

Na bazie `merged_feat`:
- dodaj `district_median_duration`,
- dodaj `duration_vs_district_median`,
- zostaw tylko te dzielnice, ktore maja co najmniej `min_trips` przejazdow.

**Wymagane metody**
- `groupby(...).transform(...)`,
- `groupby(...).filter(...)`.

Nie uzywaj `merge`-back z osobna tabelka agregatow.


In [ ]:
def duration_vs_district_profile(merged_feat: pd.DataFrame, min_trips: int = 500) -> pd.DataFrame:
    out = merged_feat.copy()
    out["district_median_duration"] = out.groupby("district")["duration_min"].transform("median")
    out["duration_vs_district_median"] = out["duration_min"] - out["district_median_duration"]
    out = out.groupby("district").filter(lambda g: len(g) >= min_trips).copy()
    return out


district_prof = duration_vs_district_profile(merged_feat, min_trips=500)
assert {"district_median_duration", "duration_vs_district_median"}.issubset(district_prof.columns)
assert district_prof.groupby("district").size().ge(500).all()
assert np.allclose(
    district_prof["duration_vs_district_median"],
    district_prof["duration_min"] - district_prof["district_median_duration"],
)
district_prof.head()


## 03.09 - Pivot Tables


### Cwiczenie 3.09.1 - Macierz: weekday x hour

Zbuduj pivot table z liczba przejazdow:
- wiersze: `weekday`
- kolumny: `hour`
- wartosci: liczba przejazdow

Ustal kolejnosc dni: `Mon, Tue, Wed, Thu, Fri, Sat, Sun`.


In [ ]:
def weekday_hour_pivot(trips: pd.DataFrame) -> pd.DataFrame:
    out = trips.copy()
    out["weekday"] = pd.Categorical(
        pd.to_datetime(out["start_ts"]).dt.day_name().str[:3],
        categories=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
        ordered=True,
    )
    out["hour"] = pd.to_datetime(out["start_ts"]).dt.hour
    return (
        out.pivot_table(
            index="weekday",
            columns="hour",
            values="trip_id",
            aggfunc="size",
            fill_value=0,
            observed=False,
        )
        .reindex(index=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], fill_value=0)
        .reindex(columns=range(24), fill_value=0)
    )


pivot = weekday_hour_pivot(trips_small)
assert isinstance(pivot, pd.DataFrame)
assert pivot.shape[1] <= 24
assert pivot.to_numpy().sum() == len(trips_small)
pivot


### Cwiczenie 3.09.2 - Pivot wielopoziomowy: `fill_value` i `margins`

Zbuduj pivot z przychodem szacowanym (`revenue_est`):
- `index=['district', 'user_type']`
- `columns='weekday'`
- `values='revenue_est'`
- `aggfunc=['sum', 'mean']`
- `fill_value=0`
- `margins=True`

**Wymagane metody**
- `pivot_table`,
- `fill_value`,
- `margins`.


In [ ]:
def revenue_pivot_multi(merged_feat: pd.DataFrame) -> pd.DataFrame:
    out = merged_feat.copy()
    out["weekday"] = pd.Categorical(
        pd.to_datetime(out["start_ts"]).dt.day_name().str[:3],
        categories=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
        ordered=True,
    )

    table = out.pivot_table(
        index=["district", "user_type"],
        columns="weekday",
        values="revenue_est",
        aggfunc=["sum", "mean"],
        fill_value=0,
        margins=True,
        observed=False,
    )

    desired_cols = pd.MultiIndex.from_tuples(
        [(agg, day) for agg in ["sum", "mean"] for day in ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun", "All"]],
        names=table.columns.names,
    )
    return table.reindex(columns=desired_cols).fillna(0)


rev_pivot = revenue_pivot_multi(merged_feat)
assert isinstance(rev_pivot.index, pd.MultiIndex)
assert isinstance(rev_pivot.columns, pd.MultiIndex)
assert ("sum", "All") in rev_pivot.columns
assert ("All", "") in rev_pivot.index
rev_pivot.head()


## 03.10 - Working With Strings


### Cwiczenie 3.10.1 - Oczysc nazwy stacji i kupony

Napisz funkcje, ktora:
- tworzy kolumne `station_name_clean` w `stations`,
- tworzy kolumne `coupon_clean` w `trips`,
- obcina spacje, zamienia litery na male, a brakujace kupony wpisuje jako `"none"`.


In [ ]:
def clean_text_columns(trips: pd.DataFrame, stations: pd.DataFrame):
    trips2 = trips.copy()
    stations2 = stations.copy()

    trips2["coupon_clean"] = (
        trips2["coupon_code"]
        .fillna("none")
        .astype(str)
        .str.strip()
        .str.lower()
        .replace("", "none")
    )
    stations2["station_name_clean"] = (
        stations2["station_name"]
        .astype(str)
        .str.strip()
        .str.lower()
    )
    return trips2, stations2


trips_txt, stations_txt = clean_text_columns(trips_small, stations_small)
assert "coupon_clean" in trips_txt.columns
assert "station_name_clean" in stations_txt.columns
assert trips_txt["coupon_clean"].isna().sum() == 0
trips_txt[["coupon_code", "coupon_clean"]].head(), stations_txt[["station_name", "station_name_clean"]].head()


### Cwiczenie 3.10.2 ⭐ - Regex: wyciagnij kod i dzielnice z nazwy stacji

Nazwy stacji maja trzy formaty:
- `st_00 | praga`
- `ST-01 / MOKOTOW`
- `Station 02 - Wola`

Wyciagnij z `station_name`:
- `station_code` (int),
- `district_from_name` (lowercase),
- `name_pattern` (`'pipe'`, `'slash'`, `'dash'`).

**Wymagane metody**
- `.str.extract(...)`,
- `.str.contains(...)`,
- `.str.lower()`.


In [ ]:
def parse_station_name_tokens(stations: pd.DataFrame) -> pd.DataFrame:
    stations2 = stations.copy()

    extracted = stations2["station_name"].str.extract(
        r"(?i)(?:st(?:ation)?)[\s_-]*(\d{2}).*?([A-Za-z]+)\s*$"
    )

    stations2["station_code"] = extracted[0].astype(int)
    stations2["district_from_name"] = extracted[1].str.lower()

    stations2["name_pattern"] = np.select(
        [
            stations2["station_name"].str.contains(r"\|", regex=True),
            stations2["station_name"].str.contains(r"/", regex=True),
        ],
        ["pipe", "slash"],
        default="dash",
    )

    return stations2


stations_parsed = parse_station_name_tokens(stations_small)
assert {"station_code", "district_from_name", "name_pattern"}.issubset(stations_parsed.columns)
assert stations_parsed["station_code"].dtype.kind in "iu"
assert stations_parsed["district_from_name"].equals(stations_parsed["district"].str.lower())
assert set(stations_parsed["name_pattern"].unique()).issubset({"pipe", "slash", "dash"})
stations_parsed[["station_name", "station_code", "district_from_name", "name_pattern"]].head(8)


## 03.11 - Working with Time Series


### Cwiczenie 3.11.1 - Dzienna liczba przejazdow i srednia 7D

Zbuduj DataFrame indeksowany po dacie z kolumnami:
- `n_trips`
- `rolling_7d`

Wskazowka: `resample('D')`.


In [ ]:
def daily_counts_with_rolling(trips: pd.DataFrame) -> pd.DataFrame:
    daily = (
        trips.set_index(pd.to_datetime(trips["start_ts"]))
        .resample("D")
        .size()
        .rename("n_trips")
        .to_frame()
    )
    daily["rolling_7d"] = daily["n_trips"].rolling(7, min_periods=1).mean()
    return daily


daily = daily_counts_with_rolling(trips_small)
assert isinstance(daily.index, pd.DatetimeIndex)
assert {"n_trips", "rolling_7d"}.issubset(daily.columns)
assert daily["n_trips"].sum() == len(trips_small)
daily.head(10)


### Mini-case końcowy — napisz 3 krótkie obserwacje z danych

To jest mały most do projektu końcowego.

Na podstawie obiektów, które już policzyłeś/aś (`pivot`, `kpis`, `daily`, `merged_feat`),
napisz trzy krótkie obserwacje:
1. kiedy ruch wygląda na największy,
2. czy bardziej widać rytm godzinowy czy różnice między dniami,
3. która tabela / transformacja była do tego najwygodniejsza i dlaczego.

Nie chodzi o „jedyną poprawną odpowiedź”, tylko o to, żeby student ćwiczył
**przechodzenie od tabeli do wniosku**.


In [ ]:
OBSERVATION_1 = (
    "Największy ruch widać w godzinach popołudniowych, szczególnie około 17:00–18:00, "
    "a drugi wyraźny pik pojawia się rano około 7:00–8:00."
)
OBSERVATION_2 = (
    "Silniejszy jest rytm godzinowy niż różnice między dniami tygodnia: profile godzin mają "
    "bardzo wyraźne piki, podczas gdy sumy dzienne są do siebie dużo bardziej zbliżone."
)
METHOD_NOTE = (
    "Najwygodniejsza była tabela `pivot` po `(weekday, hour)`, bo w jednym miejscu pokazuje oba "
    "wymiary naraz i pozwala od razu porównać kształt profilu w ciągu dnia z różnicami między dniami. "
    "Uzupełniająco przydały się `daily` i `rolling_7d`, bo dobrze streszczają zmiany dzień po dniu, "
    "ale to właśnie pivot najłatwiej zamienia surowe logi przejazdów w krótki wniosek analityczny."
)

assert isinstance(OBSERVATION_1, str) and len(OBSERVATION_1.strip()) >= 40
assert isinstance(OBSERVATION_2, str) and len(OBSERVATION_2.strip()) >= 40
assert isinstance(METHOD_NOTE, str) and len(METHOD_NOTE.strip()) >= 100
print("Mini-case analityczny ✅")


### Cwiczenie 3.11.2 ⭐ - Szereg godzinowy: `shift` i porownanie do poprzedniej doby

Zbuduj godzinowy szereg popytu z kolumnami:
- `n_trips`
- `lag_24h`
- `rolling_24h_mean`
- `vs_prev_day`

gdzie `vs_prev_day = n_trips / lag_24h`.

**Wymagane metody**
- indeks czasowy,
- `resample('h')`,
- `shift(24)`,
- `rolling(24)`.


In [ ]:
def hourly_demand_lag_features(trips: pd.DataFrame) -> pd.DataFrame:
    hourly = (
        trips.set_index(pd.to_datetime(trips["start_ts"]))
        .resample("h")
        .size()
        .rename("n_trips")
        .to_frame()
    )
    hourly["lag_24h"] = hourly["n_trips"].shift(24)
    hourly["rolling_24h_mean"] = hourly["n_trips"].rolling(24, min_periods=1).mean()
    hourly["vs_prev_day"] = hourly["n_trips"] / hourly["lag_24h"].replace(0, np.nan)
    return hourly


hourly = hourly_demand_lag_features(trips_small)
assert isinstance(hourly.index, pd.DatetimeIndex)
assert {"n_trips", "lag_24h", "rolling_24h_mean", "vs_prev_day"}.issubset(hourly.columns)
assert hourly["n_trips"].sum() == len(trips_small)
assert hourly["lag_24h"].iloc[:24].isna().all()
hourly.loc["2024-05-03"].head()


## 03.12 - Performance: eval() i query() (rozszerzenie praktyczne)


### Cwiczenie 3.12.1 - Szybki filtr po wielu warunkach

Na bazie `merged_feat` odfiltruj przejazdy, ktore spelniaja jednoczesnie:
- `duration_min >= 10`
- `duration_min <= 45`
- `precip_mm <= 1.0`
- `user_type == 'member'`

Zaimplementuj to przez `query()` lub `eval()`.


In [ ]:
def fast_member_filter(merged_feat: pd.DataFrame) -> pd.DataFrame:
    return merged_feat.query(
        "10 <= duration_min <= 45 and precip_mm <= 1.0 and user_type == 'member'"
    ).copy()


flt = fast_member_filter(merged_feat)
assert isinstance(flt, pd.DataFrame)
assert (flt["duration_min"] >= 10).all()
assert (flt["duration_min"] <= 45).all()
assert (flt["precip_mm"] <= 1.0).all()
assert (flt["user_type"] == "member").all()
flt.head()


### Cwiczenie 3.12.2 ⭐ - Mini-benchmark: petla vs wektoryzacja

Porownaj czas policzenia przychodu (`revenue`) dwoma sposobami:
- wolna funkcja `revenue_loop(...)` z pliku core,
- wersja wektorowa w NumPy/Pandas.

Zwracaj `Series` z:
- `loop_sec`
- `vectorized_sec`
- `speedup_x`

**Wymagane metody**
- `time_one`,
- wektoryzacja na kolumnach / tablicach.


In [ ]:
def benchmark_revenue_vectorization(trips: pd.DataFrame) -> pd.Series:
    duration_min = (
        pd.to_datetime(trips["end_ts"]) - pd.to_datetime(trips["start_ts"])
    ).dt.total_seconds() / 60.0
    user_type = trips["user_type"]

    def revenue_vectorized(duration_min: pd.Series, user_type: pd.Series) -> np.ndarray:
        is_member = user_type.eq("member").to_numpy()
        base = np.where(is_member, 1.0, 2.5)
        per_min = np.where(is_member, 0.12, 0.23)
        return base + per_min * np.maximum(duration_min.to_numpy() - 5.0, 0.0)

    loop_sec = time_one(revenue_loop, duration_min.to_numpy(), user_type.to_numpy(), repeat=5)
    vectorized_sec = time_one(revenue_vectorized, duration_min, user_type, repeat=5)

    return pd.Series(
        {
            "loop_sec": loop_sec,
            "vectorized_sec": vectorized_sec,
            "speedup_x": loop_sec / vectorized_sec,
        }
    )


bench_rev = benchmark_revenue_vectorization(trips_small)
assert {"loop_sec", "vectorized_sec", "speedup_x"}.issubset(bench_rev.index)
assert bench_rev["loop_sec"] > 0
assert bench_rev["vectorized_sec"] > 0
assert bench_rev["speedup_x"] >= 1
bench_rev


### Cwiczenie 3.12.3 ⭐ - Mini-benchmark: maska boolean vs `query()`

Porownaj dwa sposoby filtrowania tych samych danych:
- klasyczna maska boolean,
- `DataFrame.query(...)`.

Uzyj tych samych warunkow co w Cwiczeniu 3.12.1.

Zwracaj `Series` z:
- `mask_sec`
- `query_sec`
- `same_n_rows`

**Wymagane metody**
- maska boolean,
- `query`,
- `time_one`.


In [ ]:
def benchmark_mask_vs_query(merged_feat: pd.DataFrame) -> pd.Series:
    expr = "10 <= duration_min <= 45 and precip_mm <= 1.0 and user_type == 'member'"

    def mask_filter(df: pd.DataFrame) -> pd.DataFrame:
        mask = (
            df["duration_min"].between(10, 45)
            & df["precip_mm"].le(1.0)
            & df["user_type"].eq("member")
        )
        return df.loc[mask]

    def query_filter(df: pd.DataFrame) -> pd.DataFrame:
        return df.query(expr)

    mask_sec = time_one(mask_filter, merged_feat, repeat=5)
    query_sec = time_one(query_filter, merged_feat, repeat=5)
    same_n_rows = len(mask_filter(merged_feat)) == len(query_filter(merged_feat))

    return pd.Series(
        {
            "mask_sec": mask_sec,
            "query_sec": query_sec,
            "same_n_rows": same_n_rows,
        }
    )


bench_filter = benchmark_mask_vs_query(merged_feat)
assert {"mask_sec", "query_sec", "same_n_rows"}.issubset(bench_filter.index)
assert bool(bench_filter["same_n_rows"]) is True
bench_filter


## ⭐ Add-on - przygotowanie do projektu


### Cwiczenie A.1 - Porownaj pamiec: `object` vs `category`

Dla kolumn `user_type` i `district` porownaj zuzycie pamieci przed i po konwersji na `category`.
Zapisz krotki wniosek tekstowy w zmiennej `MEMORY_NOTE`.


In [ ]:
merged_cat = merged_feat.copy()
before = merged_cat[["user_type", "district"]].memory_usage(deep=True).sum()

merged_cat["user_type"] = merged_cat["user_type"].astype("category")
merged_cat["district"] = merged_cat["district"].astype("category")
after = merged_cat[["user_type", "district"]].memory_usage(deep=True).sum()

MEMORY_NOTE = (
    f"Po konwersji kolumn tekstowych na `category` zużycie pamięci spadło z {before} do {after} bajtów. "
    "Dzieje się tak dlatego, że Pandas przechowuje słownik unikalnych etykiet i lekkie kody liczbowe "
    "zamiast powielać pełne napisy w każdym wierszu. "
    "To szczególnie opłaca się dla kolumn z niewielką liczbą powtarzających się wartości, takich jak "
    "`user_type` i `district`."
)

assert after is not None and after <= before
assert isinstance(MEMORY_NOTE, str) and len(MEMORY_NOTE.strip()) >= 80
before, after, MEMORY_NOTE


## Do oddania

1. Uzupelnione funkcje.  
2. Krotki komentarz metodologiczny:
   - gdzie swiadomie uzyles alignmentu po indeksie zamiast `merge`,
   - gdzie potrzebne bylo `transform`, a nie `agg`,
   - po co sa `validate` i `indicator` w merge.  
3. Wyniki 3 mini-benchmarkow:
   - petla vs wektoryzacja,
   - maska boolean vs `query()`,
   - pamiec `object` vs `category`.  
4. Krotki wniosek (3-5 zdan): kiedy w praktyce wolisz `merge`, a kiedy prace po indeksie / `join` / alignment.


### Rekomendacja dla prowadzącego
Na pierwszym przejściu przez materiał nie trzeba wymagać wszystkiego naraz.
Dobrze działa model:
- CORE na zajęciach,
- 1-2 zadania z gwiazdką jako homework,
- mini-case analityczny jako pomost do projektu.
